In [20]:
import cv2, torch, os, timm, time, warnings, gc, zipfile, telegram, torchvision.transforms, pandas, numpy
from tqdm import tqdm
from glob import glob
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

device = torch.device('cuda:0')
warnings.filterwarnings('ignore')

In [3]:
label_df = pandas.read_csv('./seoul_dataset/train.csv')

In [4]:
def get_train_data(data_dir):
    img_path_list = []
    label_list = []
    img_path_list.extend(glob(os.path.join(data_dir, '*.PNG')))
    img_path_list.sort(key=lambda x: int(x.split('/')[-1].split('.')[0]))
    label_list.extend(label_df['label'])

    return img_path_list, label_list

def get_test_data(data_dir):
    img_path_list = []
    img_path_list.extend(glob(os.path.join(data_dir, '*.PNG')))
    img_path_list.sort(key=lambda x: int(x.split('/')[-1].split('.')[0]))

    return img_path_list

In [5]:
train_path, train_label = get_train_data('./seoul_dataset/train')
test_path = get_test_data('./seoul_dataset/test')

In [6]:
label_unique = sorted(numpy.unique(train_label))
label_unique = {key: value for key, value in zip(label_unique, range(len(label_unique)))}
train_labels = [label_unique[k] for k in train_label]

In [8]:
data_dir = './seoul_dataset/'

In [9]:
def img_load(path):
    img = cv2.imread(path)[:, :, ::-1]
    return img

In [10]:
train_imgs = [img_load(m) for m in tqdm(train_path)]
test_imgs = [img_load(i) for i in tqdm(test_path)]
numpy.save(data_dir + 'train_imgs', numpy.array(train_imgs))
numpy.save(data_dir + 'test_imgs', numpy.array(test_imgs))
train_imgs = numpy.load(data_dir + 'train_imgs.npy')
test_imgs = numpy.load(data_dir + 'test_imgs.npy')

100%|██████████| 199/199 [00:03<00:00, 52.01it/s]


In [22]:
def f1_score_function(real, pred):
    score = accuracy_score(real, pred)
    return score

In [23]:
class Custom_dataset(Dataset):
    def __init__(self, img_paths, labels, mode='train'):
        self.img_paths = img_paths
        self.labels = labels
        self.mode = mode

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img = self.img_paths[idx]
        if self.mode == 'train':
            train_transform = torchvision.transforms.Compose([
                torchvision.transforms.ToTensor(),
                torchvision.transforms.Resize((224, 224)),
                torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            ])
            img = train_transform(img)
        if self.mode == 'test':
            test_transform = torchvision.transforms.Compose([
                torchvision.transforms.ToTensor(),
                torchvision.transforms.Resize((224, 224)),
                torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            ])
            img = test_transform(img)

        label = self.labels[idx]
        return img, label

In [24]:
class Network(torch.nn.Module):
    def __init__(self, mode='train'):
        super(Network, self).__init__()
        self.mode = mode
        if self.mode == 'train':
            self.model = timm.create_model(model_name, pretrained=True, num_classes=10)
        if self.mode == 'test':
            self.model = timm.create_model(model_name, pretrained=True, num_classes=10)

    def forward(self, x):
        x = self.model(x)
        return x

In [25]:
batch_size = 128
epochs = 100
model_name = "densenet121"

In [26]:
cv = StratifiedKFold(n_splits=5, random_state=2022, shuffle=True)
pred_ensemble = []

for idx, (train_idx, val_idx) in enumerate(cv.split(train_imgs, numpy.array(train_labels))):
    print("-----------------fold_{} start!----------------".format(idx))
    t_imgs, val_imgs = train_imgs[train_idx], train_imgs[val_idx]
    t_labels, val_labels = numpy.array(train_labels)[train_idx], numpy.array(train_labels)[val_idx]

    train_dataset = Custom_dataset(numpy.array(t_imgs), numpy.array(t_labels), mode='train')
    train_loader = DataLoader(train_dataset, shuffle=True, batch_size=batch_size)
    val_dataset = Custom_dataset(numpy.array(val_imgs), numpy.array(val_labels), mode='test')
    val_loader = DataLoader(val_dataset, shuffle=True, batch_size=batch_size)

    gc.collect()
    torch.cuda.empty_cache()
    best = 0
    model = Network().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-3)
    criterion = torch.nn.CrossEntropyLoss()
    scaler = torch.cuda.amp.GradScaler()
    best_f1 = 0
    early_stopping = 0

    for epoch in range(epochs):
        start = time.time()
        train_loss = 0
        train_pred = []
        train_y = []
        model.train()
        for batch in (train_loader):
            optimizer.zero_grad()
            x = torch.tensor(batch[0], dtype=torch.float32, device=device)
            y = torch.tensor(batch[1], dtype=torch.long, device=device)
            with torch.cuda.amp.autocast():
                pred = model(x)
            loss = criterion(pred, y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item() / len(train_loader)
            train_pred += pred.argmax(1).detach().cpu().numpy().tolist()
            train_y += y.detach().cpu().numpy().tolist()
        train_f1 = f1_score_function(train_y, train_pred)
        state_dict = model.state_dict()
        model.eval()
        with torch.no_grad():
            val_loss = 0
            val_pred = []
            val_y = []

            for batch in (val_loader):
                x_val = torch.tensor(batch[0], dtype=torch.float32, device=device)
                y_val = torch.tensor(batch[1], dtype=torch.long, device=device)
                with torch.cuda.amp.autocast():
                    pred_val = model(x_val)
                loss_val = criterion(pred_val, y_val)

                val_loss += loss_val.item() / len(val_loader)
                val_pred += pred_val.argmax(1).detach().cpu().numpy().tolist()
                val_y += y_val.detach().cpu().numpy().tolist()
            val_f1 = f1_score_function(val_y, val_pred)

            if val_f1 > best_f1:
                best_epoch = epoch
                best_loss = val_loss
                best_f1 = val_f1
                early_stopping = 0
                torch.save({'epoch': epoch,
                            'state_dict': state_dict,
                            'optimizer': optimizer.state_dict(),
                            'scaler': scaler.state_dict(),
                            }, data_dir + model_name + '-best_model_{}.pth'.format(idx))
                print('-----------------SAVE:{} epoch----------------'.format(best_epoch + 1))
            else:
                early_stopping += 1
            # Early Stopping
            if early_stopping == 20:
                TIME = time.time() - start
                print(
                    f'Epoch : {epoch + 1}/{epochs}, time : {TIME:.0f}s/{TIME * (epochs - epoch - 1):.0f}s, Train loss : {train_loss:.5f}, f1 : {train_f1:.5f}, Val loss : {val_loss:.5f}, f1 : {val_f1:.5f}')
                print("Early stopping")
                break

        TIME = time.time() - start
        print(
            f'Epoch : {epoch + 1}/{epochs}, time : {TIME:.0f}s/{TIME * (epochs - epoch - 1):.0f}s, Train loss : {train_loss:.5f}, f1 : {train_f1:.5f}, Val loss : {val_loss:.5f}, f1 : {val_f1:.5f}')

-----------------fold_0 start!----------------
-----------------SAVE:1 epoch----------------
Epoch : 1/100, time : 3s/337s, Train loss : 2.20957, f1 : 0.18166, Val loss : 1.95654, f1 : 0.37241
-----------------SAVE:2 epoch----------------
Epoch : 2/100, time : 4s/348s, Train loss : 1.45625, f1 : 0.74221, Val loss : 1.49756, f1 : 0.75862
-----------------SAVE:3 epoch----------------
Epoch : 3/100, time : 4s/343s, Train loss : 0.93115, f1 : 0.95675, Val loss : 0.87476, f1 : 0.93103
-----------------SAVE:4 epoch----------------
Epoch : 4/100, time : 4s/386s, Train loss : 0.60967, f1 : 0.99308, Val loss : 0.66699, f1 : 0.97241
-----------------SAVE:5 epoch----------------
Epoch : 5/100, time : 4s/336s, Train loss : 0.38081, f1 : 0.99827, Val loss : 0.47656, f1 : 0.98621
-----------------SAVE:6 epoch----------------
Epoch : 6/100, time : 4s/340s, Train loss : 0.24111, f1 : 0.99827, Val loss : 0.40356, f1 : 0.99310
Epoch : 7/100, time : 3s/303s, Train loss : 0.16079, f1 : 1.00000, Val loss :

-----------------SAVE:12 epoch----------------
Epoch : 12/100, time : 4s/311s, Train loss : 0.03389, f1 : 1.00000, Val loss : 0.14398, f1 : 0.99310
Epoch : 13/100, time : 3s/297s, Train loss : 0.02927, f1 : 1.00000, Val loss : 0.11285, f1 : 0.99310
Epoch : 14/100, time : 3s/285s, Train loss : 0.02624, f1 : 1.00000, Val loss : 0.13348, f1 : 0.99310
Epoch : 15/100, time : 3s/272s, Train loss : 0.02215, f1 : 1.00000, Val loss : 0.09137, f1 : 0.99310
Epoch : 16/100, time : 3s/275s, Train loss : 0.01833, f1 : 1.00000, Val loss : 0.07930, f1 : 0.99310
Epoch : 17/100, time : 3s/278s, Train loss : 0.01874, f1 : 1.00000, Val loss : 0.11752, f1 : 0.99310
Epoch : 18/100, time : 3s/275s, Train loss : 0.01562, f1 : 1.00000, Val loss : 0.07117, f1 : 0.99310
Epoch : 19/100, time : 6s/510s, Train loss : 0.01723, f1 : 1.00000, Val loss : 0.06323, f1 : 0.99310
Epoch : 20/100, time : 3s/256s, Train loss : 0.01287, f1 : 1.00000, Val loss : 0.06483, f1 : 0.99310
Epoch : 21/100, time : 3s/254s, Train loss :

Epoch : 27/100, time : 3s/231s, Train loss : 0.00967, f1 : 1.00000, Val loss : 0.05612, f1 : 1.00000
Epoch : 28/100, time : 3s/225s, Train loss : 0.00905, f1 : 1.00000, Val loss : 0.03972, f1 : 1.00000
Epoch : 29/100, time : 3s/225s, Train loss : 0.00881, f1 : 1.00000, Val loss : 0.03655, f1 : 1.00000
Epoch : 30/100, time : 3s/219s, Train loss : 0.00799, f1 : 1.00000, Val loss : 0.03696, f1 : 1.00000
Epoch : 31/100, time : 3s/216s, Train loss : 0.00722, f1 : 1.00000, Val loss : 0.03980, f1 : 1.00000
Epoch : 32/100, time : 4s/246s, Train loss : 0.00717, f1 : 1.00000, Val loss : 0.07611, f1 : 1.00000
Epoch : 33/100, time : 3s/214s, Train loss : 0.00630, f1 : 1.00000, Val loss : 0.03686, f1 : 1.00000
Epoch : 34/100, time : 3s/215s, Train loss : 0.00603, f1 : 1.00000, Val loss : 0.06949, f1 : 1.00000
Early stopping


In [27]:
pred_ensemble = []
test_dataset = Custom_dataset(numpy.array(test_imgs), numpy.array(["tmp"] * len(test_imgs)), mode='test')
test_loader = DataLoader(test_dataset, shuffle=False, batch_size=batch_size)
for i in range(5):
    model_test = Network(mode='test').to(device)
    model_test.load_state_dict(torch.load((data_dir + model_name + '-best_model_{}.pth'.format(i)))['state_dict'])
    model_test.eval()
    pred_prob = []
    with torch.no_grad():
        for batch in (test_loader):
            x = torch.tensor(batch[0], dtype=torch.float32, device=device)
            with torch.cuda.amp.autocast():
                pred = model_test(x)
                pred_prob.extend(pred.detach().cpu().numpy())
        pred_ensemble.append(pred_prob)

In [28]:
pred = (numpy.array(pred_ensemble[0]) + numpy.array(pred_ensemble[1]) + numpy.array(pred_ensemble[2]) + numpy.array(
    pred_ensemble[3]) + numpy.array(pred_ensemble[4])) / 5
f_pred = numpy.array(pred).argmax(1).tolist()
label_decoder = {val: key for key, val in label_unique.items()}
f_result = [label_decoder[result] for result in f_pred]

In [29]:
submission = pandas.read_csv('./seoul_dataset/sample_submission.csv')
submission['label'] = f_result

In [32]:
submission.to_csv('./seoul_dataset/submit.csv', index=False)